# Overview

After running the detection of YOLOv5-small weights from paper 1 on the downloaded sample of "GBIF" images, we get the predicted label txt files in a single folder.
The detections were run with the script `detect_insects_yolov5.sh` on the EVE cluster.

Read all the YOLO predictions and group them by family folders.

Then for each family folder, convert the YOLO coordinates to a single CSV file that the [VIA annotation tool][1] can import for visualizing and manually adjusting the bounding boxes of each insect that will be used to crop the images.

Once the csv files are computed (see the notebook below), place the via.html file in the folder `.../PAI_diptera/data/gbif_occurences/sampled/`. The via.html file used was downloaded & unzipped from [via-2.0.12.zip][2].

For each VIA csv file (each family):
- open via.html;
- menu `Annotation` > `Import Annotations (from csv)` and navigate to the csv file (e.g. Anthomyiidae_csv.csv);
- menu `Project` > `Add local files` and navigate to `.../PAI_diptera/data/gbif_occurences/sampled/images/` and select the corresponding family folder (e.g. Anthomyiidae). Open the family folder and press `CTRL + a` key combination to select all images, then click `Open`;
- Now you should be able to visualize the images and their bounding boxes predicted by YOLO.

For each family, I exported the VIA JSON project so that a research assistant manually checks the bounding boxes. The advantage of the VIA project file is that it can store the relative path to the images.

To export the VIA project:
- `Setting` button and for place the family name for the `Project Name` (e.g. Anthomyiidae);
- for `Default Path` give the relative path to the folder with the images. It is relative to where you place the via.html file. I used `./images/family_name/` (e.g `./images/Bombyliidae/`). Do nto forget to put the `/` at the very end;
- Click the `Save` button to save the settings;
- menu `Project` > `Save` > make sure all boxes are checked > `OK`. the JSON project file will be saved directly in your default downloads folder. This VIA JSON project file is different from the JSON COCO file format. The JSON project file contains the region attributes (bounding box coordinates) but not image dimensions, and the JSON COCO file contains the image dimensions, but not the regions attributes.
- to load a VIA JSON project file: menu `Project` > `Load` and the navigate to the file.

[1]: https://www.robots.ox.ac.uk/~vgg/software/via/
[2]: https://www.robots.ox.ac.uk/~vgg/software/via/downloads/via-2.0.12.zip

# Global settings

Set the imported libraries & packages, any relative paths.

In [1]:
import os
import shutil
import git

import csv
import json
import pandas as pd
import numpy as np

In [2]:
# Find the working directory of the parent repository.
# This will be used to build various data paths.
repo = git.Repo('.', search_parent_directories=True)
parent_dir = repo.working_tree_dir
print(parent_dir)

/home/vs66tavy/iDiv Dropbox/Valentin Stefan/insect-detection/pai_p3_diptera_families/PAI_diptera


# Group label files by family folders

Read all the YOLO predictions and group them by family folders.

In [3]:
# Set the path to the YOLO predicted labels folder and the output folder
labels_dir = os.path.join(parent_dir, 'architectures', 'yolov5', 'runs', 'detect', 
                          'predlbl_imgsize_640_maxdet_1000_conf_0.3_iou_0.6' , 'labels')
                            
output_dir = os.path.join(parent_dir, 'data', 'gbif_occurences', 'sampled', 
                          'yolo_predictions')

# Create the output folder if it doesn't exist
if not os.path.exists(labels_dir):
    os.makedirs(output_dir)

In [5]:
# Loop through all the files in the labels folder
for filename in os.listdir(labels_dir):
    if filename.endswith('.txt'):
        # Extract the family name from the filename.
        # The family name is the first part of the filename, separated by an underscore.
        # "Anthomyiidae_Acyglossa_atramentaria_2758828_1019547180.txt"
        # "Anthomyiidae" is the family name
        family_name = filename.split('_')[0]

        # Create the family folder if it doesn't exist
        family_folder = os.path.join(output_dir, family_name)
        if not os.path.exists(family_folder):
            os.makedirs(family_folder)

        # Move the file to the family folder
        file_path = os.path.join(labels_dir, filename)
        shutil.move(file_path, family_folder)

# YOLO to VIA conversion

For each family folder, convert the label txt files to a CSV file that can be read by VIA.

In [5]:
# def yolo_to_via_csv(yolo_lbl_dir, via_coco_dir):

# Define the families. These are the folder names created above in the output_dir
# List all directories in the output_dir
families = [f for f in os.listdir(output_dir) if os.path.isdir(os.path.join(output_dir, f))]

print(families)

['Calliphoridae', 'Tachinidae', 'Scathophagidae', 'Conopidae', 'Anthomyiidae', 'Bombyliidae', 'Stratiomyidae', 'Sarcophagidae', 'Sepsidae', 'Tabanidae', 'Muscidae', 'Empididae', 'Syrphidae', 'Fanniidae', 'Hybotidae']


Give the directory with the locations of the json COCO annotation files obtained from  the VIA annotation tool. They contain the image dimensions but not the bounding box coordinates. I obtained these files by loading the images for each family into VIA  (no annotations needed at this point), and then exporting the VIA annotations as JSON COCO format.

As an alternative, for future, I need to implement a function that extracts the image dimensions from the images themselves (using PIL or similar).

However, because the images are stored on a remote server, reading image dimensions via the internet will be slow. For now it is good enough as it is.

In [6]:
# Directory with the locations of the json COCO annotation files
via_coco_dir = os.path.join(parent_dir, 'data', 'gbif_occurences', 'sampled',
                            'via_annotations', 'coco')

print(via_coco_dir)

/home/vs66tavy/iDiv Dropbox/Valentin Stefan/insect-detection/pai_p3_diptera_families/PAI_diptera/data/gbif_occurences/sampled/via_annotations/coco


In [7]:
# Print the files in the via_coco_dir
print(os.listdir(via_coco_dir))

['Anthomyiidae_coco.json', 'Hybotidae_coco.json', 'Sepsidae_coco.json', 'Muscidae_coco.json', 'Calliphoridae_coco.json', 'Bombyliidae_coco.json', 'Sarcophagidae_coco.json', 'Empididae_coco.json', 'Stratiomyidae_coco.json', 'Fanniidae_coco.json', 'Tabanidae_coco.json', 'Scathophagidae_coco.json', 'Conopidae_coco.json', 'Anthomyiidae_csv.csv', 'Syrphidae_coco.json', 'Tachinidae_coco.json']


Loop trough each family and convert the YOLO labels to VIA annotations using the image dimensions stored in the JSON COCO files.

Note to self - I need to make a conversion function and then call that for each family so that perhaps this code is easier to read.

In [8]:
for family in families:
    yolo_lbl_dir = os.path.join(parent_dir, 'data', 'gbif_occurences', 'sampled', 
                                'yolo_predictions', family)

    # For the given family, loop through all the files in the labels folder yolo_lbl_dir 
    # and read the annotations into a data frame.
    # A annotation txt file can contain 1 or multiple lines like this 
    # (without the header and without the # symbol):
    # class  x_center_rel  y_center_rel  width_rel  height_rel  confidence
    # 2      0.428906      0.521176      0.595312   0.825882    0.804681

    # Initialize empty DataFrame to store annotations.
    columns = ['filename', 'class_id', 'x_center_rel', 'y_center_rel', 
            'width_rel', 'height_rel', 'confidence']
    df_yolo = pd.DataFrame(columns=columns)

    lst_yolo = []

    # Loop through all YOLO text files in label directory
    for filename in os.listdir(yolo_lbl_dir):
        if filename.endswith('.txt'):
            # Load annotations from YOLO text file
            with open(os.path.join(yolo_lbl_dir, filename), 'r') as f:
                annotations = [line.strip().split() for line in f]

            # Add annotations to list
            for ann in annotations:
                lst_yolo.append({
                    'filename': os.path.splitext(filename)[0],
                    'class_id': ann[0],
                    'x_center_rel': float(ann[1]),
                    'y_center_rel': float(ann[2]),
                    'width_rel': float(ann[3]),
                    'height_rel': float(ann[4]),
                    'confidence': float(ann[5])
                })

    # Convert annotations list to DataFrame
    df_yolo = pd.DataFrame(lst_yolo)


    # Read the COCO annotations file to get the image dimensions
    coco_file = os.path.join(via_coco_dir, family + '_coco.json')

    # Load COCO annotations file as a dictionary
    with open(coco_file, 'r') as f:
        coco_dict = json.load(f)

    # Get the image dimensions from the COCO annotations file
    image_dims = coco_dict['images']
    columns_coco = ['id', 'file_name', 'width', 'height']
    df_coco = pd.DataFrame(image_dims, columns=columns_coco)

    # Remove the file extension from the filename so that we can merge with the 
    # annotations DataFrame based on the filename column.
    df_coco['filename'] = df_coco['file_name'].apply(lambda x: os.path.splitext(x)[0])


    # Merge the image dimensions with the annotations DataFrame.
    # Note that while df_coco contains info about each image, df_yolo might not
    # contain info about all images. This is because YOLO might not have detected
    # any objects in an image.
    df = pd.merge(df_coco, df_yolo, on='filename', how='left')
    df = df.sort_values('filename')


    # Calculate the absolute bounding box coordinates for VIA
    df['x_min'] = (df['x_center_rel'] - df['width_rel']/2) * df['width']
    df['y_min'] = (df['y_center_rel'] - df['height_rel']/2) * df['height']
    df['x_max'] = (df['x_center_rel'] + df['width_rel']/2) * df['width']
    df['y_max'] = (df['y_center_rel'] + df['height_rel']/2) * df['height']

    df['x_min'] = np.ceil(df['x_min'])
    df['y_min'] = np.ceil(df['y_min'])
    df['x_max'] = np.ceil(df['x_max'])
    df['y_max'] = np.ceil(df['y_max'])

    df['via_x'] = df['x_min']
    df['via_y'] = df['y_min']
    df['via_width'] = df['x_max'] - df['x_min']
    df['via_height'] = df['y_max'] - df['y_min']


    # Define the columns of the output CSV file
    columns = ['filename', 'file_size', 'file_attributes', 'region_count', 
            'region_id', 'region_shape_attributes', 'region_attributes']

    # Create a new DataFrame with the desired columns
    df_via_output = pd.DataFrame(columns=columns)

    # Group the rows by file_name (with the file extension) and create a list of 
    # dictionaries for each group.
    groups = df.groupby('file_name')
    rows = []
    for filename, group in groups:
        # Reset the index of the group. This is necessary because the group is a
        # subset of the original DataFrame and the index of the group should not be
        # saved as the index of the original DataFrame.
        group = group.reset_index()
        # Iterate through each row in the group
        for idx, row in group.iterrows():
            # Create a dictionary with the desired columns
            row_dict = {
                'filename': filename,
                'file_size': np.nan,
                'file_attributes': '{}',
                # If there were no predictions (coordinates are NaN), then assign 0
                # to region_count (this is what VIA expects in such situations)
                'region_count': 0 if np.isnan(row['x_min']) else len(group),
                'region_id': idx,
                # If there were no predictions (coordinates are NaN), then assign {}
                # to region_shape_attributes & region_attributes
                'region_shape_attributes': json.dumps({}) if np.isnan(row['x_min']) 
                                                else json.dumps({
                                                    'name': 'rect',
                                                    'x': int(row['via_x']),
                                                    'y': int(row['via_y']),
                                                    'width': int(row['via_width']),
                                                    'height': int(row['via_height']),
                                                }),
                'region_attributes': json.dumps({}) if np.isnan(row['x_min'])
                                                else json.dumps({
                                                    'class_id': row['class_id'],
                                                    'confidence': row['confidence'],
                                                })
            }
            # Add the dictionary to the list of rows
            rows.append(row_dict)

    # Convert the list of rows to a DataFrame
    df_via_output = pd.DataFrame(rows)

    # Save the dataframe to a CSV file
    via_csv_file = os.path.join(via_coco_dir, family + '_csv.csv')
    df_via_output.to_csv(via_csv_file, index=False)